In [1]:
!pip install torch transformers accelerate bitsandbytes sentence-transformers faiss-cpu pypdf tqdm huggingface_hub

from IPython.display import clear_output
clear_output()

In [2]:
import os
import re
import torch
import faiss
import numpy as np
from tqdm import tqdm
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)


Logging into huggingface and downloading necessary models


In [ ]:
from huggingface_hub import login

login(token='XXX')

In [ ]:
PATH = '/data'
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
LLM_MODEL_NAME = "speakleash/Bielik-11B-v3.0-Instruct"
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
TOP_K = 3

import os
from pypdf import PdfReader
from tqdm import tqdm

def load_pdfs(folder_path):
    documents = []
    
    for filename in tqdm(os.listdir(folder_path)):
        if filename.endswith(".pdf"):
            filepath = os.path.join(folder_path, filename)
            reader = PdfReader(filepath)
            
            text = ""
            for page in reader.pages:
                text += page.extract_text() or ""
            
            documents.append({
                "source": filename,
                "text": text
            })
    
    return documents

Chunking text for better processing

In [7]:
def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
        
    return chunks

In [8]:

def prepare_chunks(pdf_docs):
    all_chunks = []
    metadata = []

    for doc in pdf_docs:
        chunks = chunk_text(doc["text"], CHUNK_SIZE, CHUNK_OVERLAP)

        for chunk in chunks:
            all_chunks.append(chunk)
            metadata.append({"source": doc["source"]})

    return all_chunks, metadata

Building vector index

In [9]:
def build_vector_index(chunks):
    print("Loading embedding model...")
    embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    print("Creating embeddings...")
    embeddings = embedding_model.encode(
        chunks,
        show_progress_bar=True
    )

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings))
    print('Done')
    return index, embedding_model

print('Executed')

Executed


In [10]:
def load_quantized_llm():
    print("Loading quantized Bielik model...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )

    tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)

    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )

    model.eval()

    return tokenizer, model

In [11]:
def retrieve(query, index, embedding_model, chunks, metadata, k=3):
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(np.array(query_embedding), k)

    results = []
    for i in indices[0]:
        results.append({
            "text": chunks[i],
            "source": metadata[i]["source"]
        })

    return results

In [12]:
def build_prompt(query, retrieved_docs):
    context = ""
    sources = set()

    for doc in retrieved_docs:
        context += doc["text"] + "\n\n"
        sources.add(doc["source"])

    prompt = f"""
Jesteś asystentem medycznym.
Odpowiadaj wyłącznie na podstawie podanego kontekstu.
Jeśli nie ma informacji w kontekście, powiedz że nie znaleziono informacji.

Kontekst:
{context}

Pytanie:
{query}

Odpowiedź:
"""
    return prompt, list(sources)

In [13]:
def generate_answer(query, index, embedding_model, chunks, metadata, tokenizer, model):
    retrieved_docs = retrieve(query, index, embedding_model, chunks, metadata, TOP_K)
    prompt, sources = build_prompt(query, retrieved_docs)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True
        )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer, sources

# Ladowanie Bielika 

In [14]:
print("Loading quantized Bielik...")
tokenizer, model = load_quantized_llm()

print("\nRAG system ready!\n")
print(model)
print(tokenizer)

Loading quantized Bielik...
Loading quantized Bielik model...


config.json:   0%|          | 0.00/710 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/143 [00:00<?, ?B/s]


RAG system ready!

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32128, 4096, padding_idx=2)
    (layers): ModuleList(
      (0-49): 50 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)

# Przetwarzanie danych

Zajmuje jakieś pół godziny dla 3 plików. Potrzebny jest mocarny komputer albo zmiana biblioteki/kodu

In [12]:
pdf_docs = load_pdfs(PATH)
chunks, metadata = prepare_chunks(pdf_docs)
index, embedding_model = build_vector_index(chunks)

100%|██████████| 3/3 [00:23<00:00,  7.94s/it]


Loading embedding model...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Creating embeddings...


Batches:   0%|          | 0/71 [00:00<?, ?it/s]

Done


In [ ]:
while True:
    query = input("\nAsk a question (or type 'exit'): ")
    if query.lower() == "exit":
        break

    answer, sources = generate_answer(
        query,
        index,
        embedding_model,
        chunks,
        metadata,
        tokenizer,
        model
    )

    print("\nANSWER:\n")
    print(answer)
    print("\nSOURCES:", sources)